# Worked solutions · Section 5 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F8.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Definition B.3.1 (`def.attributeReference`), eq. (B.7.1, `eq.commutingFold`).
Prices are integer counts of ticks, and `BUY` and `SELL` are the directions $+1$ and $-1$.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `05-inheritance` runs them and verifies every key.

# Section 5 · Classes and inheritance

**F1. Attribute references.**
Every class $C$ has a linearisation, the tuple `C.__mro__` of classes $(C_0, \dots, C_n)$,
from $C_0 = C$ to $C_n$ = `object`.
`C.name` denotes the attribute `name` of the first of $C_0, \dots, C_n$ that has one;
for an instance $x$ of $C$, `x.name` denotes the attribute of $x$ if $x$ has one,
and otherwise `C.name`, with $x$ prepended to the arguments when `C.name` is a function;
a reference that finds no attribute raises `AttributeError`,
Definition B.3.1 (`def.attributeReference`).
An assignment is not a search:
`x.name = value` binds the name in the instance, whatever the classes hold
(the second paragraph after the definition).
It creates the attribute when $x$ has none, and raises nothing,
Remark B.2.2 (`remark.misspeltAttribute`),
unless the class binds a property of that name (F2), is a frozen data class (F3),
or keeps the attributes of its instances in slots (Section B.4 (`sec.dataClasses`)).

**F2. Properties and class methods.**
For a function $f$ bound to `name` in the class $C$, Table B.2 (`tab.kindsOfMethod`):
- under `@property`, `x.name` denotes the value $f(x)$,
  computed from the instance at every reading (the paragraph after the table);
- under `@classmethod`, `x.name` and `C.name` both denote $u \mapsto f(C, u)$:
  the function receives the class on which the reference is written, or the class of $x$,
  and never the instance.

A property is found before an attribute of the instance of the same name,
and it refuses assignment, Remark B.5.1 (`remark.completeRule`);
the error Python raises is `AttributeError`.
A name bound in the body of a class to something other than a function,
such as `BAND_MARGIN`, is an attribute of the class,
read through every instance (Section B.5 (`sec.propertiesAndClassMethods`)).

**F3. Frozen data classes.**
Under `@dataclass(frozen=True)` an assignment to a field raises an error,
Section B.4 (`sec.dataClasses`): `dataclasses.FrozenInstanceError`.
The freezing is shallow:
the fields cannot be rebound, and the objects they refer to keep their own mutability,
so the list of fills of a `SubmitResult` can be appended to,
Remark B.4.1 (`remark.frozenIsShallow`).

**F4. Inheritance and late binding.**
`class D(B)` gives $D$ the linearisation $D$ followed by that of $B$:
a name that $D$ binds overrides the one of $B$, and a name it does not bind is inherited,
Section B.7 (`sec.inheritance`).
In a function of $B$ called on an instance $x$ of $D$,
every reference `self.g` is `x.g`, resolved along the linearisation of $D$,
Proposition B.7.1 (`prop.lateBinding`): an inherited method calls the overrides.
Inside a function of the class $C_k$ of a linearisation,
`super().name` begins the search at $C_{k+1}$: it is how an override calls the function it overrides.
The attributes of an instance are bound by the constructors that run,
Remark B.7.2 (`remark.constructorsBindTheState`).

**F5. The storage and the matching.**
`AggregateBook` keeps each side as a dictionary from price to size, `bids` and `asks`,
and everything it does is written against four operations,
Section 1.2.3 (`sec.foldImplementation`):
`levels_map`, the dictionary of a side;
`size_at`, the size at a price, 0 where nothing rests;
`set_size`, which writes a size and removes the price when the size is 0;
and `best_price`, the `max` of the bid prices or the `min` of the ask prices, `None` on an empty side,
at a cost $O(\ell)$ for $\ell$ occupied prices, Table 1.4 (`tab.fasterBookCosts`).
`apply` passes a submission to `submit` and a withdrawal to `withdraw`,
which writes the reduced size through `set_size`.
`submit`, Listing 1.2 (`listing.submit`), looks up `best_price(-direction)` at the top of every pass of its loop.
It breaks when that side is empty or its best price is not eligible,
`best * direction > limit_price * direction`;
otherwise it trades `min(remaining, resting)` there, writes what is left through `set_size`,
and records the price as `last_fill_price`.
The loop ends when nothing remains, without a further lookup.
What is left of the order rests at its limit price;
a market order carries a sentinel price, and rests what is left at `last_fill_price`,
the market-to-limit rule, the third departure from Algorithm 1.2.1 (`algo.bookFold`)
described in Section 1.2.3 (`sec.foldImplementation`).
`last_traded_size` is the number of shares the last message traded.

**F6. The cache of `CachedBestBook`.**
`CachedBestBook` keeps the two dictionaries and, beside them, the best price of each side in `_cached`,
Listing B.11 (`listing.cachedBestBook`).
Its `set_size` writes the dictionary through `super().set_size`
and then sets the cache to `_best_after(direction, price, size)`,
which `unito26.lob.orderbook.CachedBestBook._best_after` defines by three cases:
the written price, if it is occupied and the cache is `None` or worse;
a rescan of the dictionary by `self._rescan`, if the write empties the cached price;
and the cache unchanged otherwise.
Its `best_price` returns the cache, unchecked.
The cache is the best price of the dictionary, the third invariant of Section B.6 (`sec.stateAndInvariants`),
by Proposition B.6.2 (`prop.invariantByInduction`): the constructor starts it at `None` on empty sides (i),
the three cases keep it at every write (ii), and every write goes through `set_size` (iii).
A write past `set_size` breaks iii and can leave the cache behind, Listing B.10 (`listing.silentCache`).

**F7. Building a book, and the band.**
`from_levels` creates the book by `cls.for_prices(list(bids) + list(asks), strict)`,
writes every level through `set_size`,
and ends with `check_invariants`, Listing B.8 (`listing.fromLevels`).
`AggregateBook.check_invariants` reads `best_bid_price` and `best_ask_price` once each,
and each is a call of `self.best_price`.
`AggregateBook.for_prices` returns `cls(strict)`.
`TickArrayBook.for_prices` computes, from the lowest and the highest price and `cls.BAND_MARGIN`,
the floor $p_0 = \text{low} - \mathtt{BAND\_MARGIN}$
and the width $W = \text{high} - \text{low} + 1 + 2\,\mathtt{BAND\_MARGIN}$;
the ceiling is $p_0 + W - 1$, and `set_size` raises `ValueError` at a price outside $[p_0, p_0 + W - 1]$.
`BitmapBook.for_prices` computes the floor alone, and its `set_size` refuses a price below it.
`AggregateBook` binds `BAND_MARGIN = 64`, the margin of 64 ticks on either side of Section 1.2.4 (`sec.fasterBook`).

**F8. Calls.**
Calling a class creates a new instance of it, and `type(x)` is the class of `x`,
Definition B.2.1 (`def.classAndInstance`);
the call `C(u)` runs `C.__init__(x, u)` on the new instance $x$, Table B.1 (`tab.operationsAsCalls`).
A fact of Python: a call binds its positional arguments to the parameters in order,
and raises `TypeError` when a parameter without a default is left without a value.

## 5.1 · `question.writesToABookAndAMessage`

The book `AggregateBook.from_levels({99: 100}, {101: 100})`, the limit buy of 50 at 100,
and five statements, each run on its own after them.

**Answer: (A).** (2), (3) and (5) raise, and no other.

- **(1)** `book.best_bid = 100`.
  No class binds `best_bid`, and `AggregateBook` declares no `__slots__`,
  so the assignment creates an attribute of the instance (F1), which no function of the class reads.
  No error.
- **(2)** `book.spread = 1`.
  `spread` is a property, and a property refuses assignment (F2): `AttributeError`.
- **(3)** `book.spread()`.
  The reading `book.spread` is already the value of the property (F2), the integer $101 - 99 = 2$,
  and calling an integer raises `TypeError`, a fact of Python.
- **(4)** `book.bids = {100: 50}`.
  The constructor bound `bids` on the instance, and the assignment rebinds it (F1). No error.
- **(5)** `message.size = 0`.
  `Message` is a frozen data class and `size` is one of its fields (F3): `FrozenInstanceError`.

**The other choices.**
- (B) (1) as well: takes the assignment for a reference, which raises `AttributeError` when it finds nothing;
  an assignment searches nothing (F1).
- (C) (3) and (5) only: takes `book.spread = 1` to bind an attribute of the instance, as (1) does;
  the property is found first, and refuses it (F2).
- (D) (2) and (5) only: sees that the property refuses assignment,
  and takes the parentheses of `book.spread()` to call the function under `@property`, as they call a method;
  the reading `book.spread` is already the value $f(x)$ (F2).
- (E) (4) as well: takes the attributes bound by the constructor to be fixed, as the fields of a frozen data class are;
  `AggregateBook` is a plain class.

## 5.2 · `question.frozenSubmitResultHoldsAList`

Bid 99 × 100; asks 101 × 100 and 102 × 80.
A market buy of 150, applied with `record=True`; one fill appended to the result, then the fills rebound.

**Answer: (C).** It prints `250 150`, then raises `FrozenInstanceError`.

**The fills.**
The buy takes the 100 at 101 and 50 of the 80 at 102 (F5):
`result.fills` holds a fill of 100 at 101 and one of 50 at 102,
and `book.last_traded_size` is 150.

**The append.**
`SubmitResult` is frozen, and the list that its field `fills` refers to is not (F3):
the append succeeds, and the list holds the fill of 100 at 101 twice.
`market_order_size` is a property, computed from the list at every reading (F2): $100 + 50 + 100 = 250$.
Nothing tells the book of the append, and `last_traded_size` is still 150.
The first line is `250 150`.

**The rebinding.**
`result.fills = result.fills[:1]` assigns to a field of a frozen instance,
and raises `FrozenInstanceError` (F3).
The second `print` is never reached.

In [ ]:
at_101 = min(150, 100)
at_102 = min(150 - at_101, 80)
at_101 + at_102, at_101 + at_102 + at_101

**The other choices.**
- (A) `FrozenInstanceError` at the append: takes the freezing to reach the list the field refers to;
  it is shallow (F3).
- (B) `150 150`, then the error: takes `market_order_size` for a value stored when the result was made;
  a property is computed at its reading (F2).
- (D) `250 150`, then `100`: misses that the frozen instance refuses the rebinding of `fills`;
  had it been allowed, the one fill left would sum to 100.

## 5.3 · `question.bandMarginShadowedOnTheInstance`

The prices 1000 and 1010; `AggregateBook` binds `BAND_MARGIN = 64`.
A margin of 0 bound on an instance of `TickArrayBook`, and one bound on its subclass `NarrowBook`.

**Answer: (B).** `936 139 1000 11`.

**`wide`.**
`TickArrayBook.for_prices` receives `TickArrayBook` (F2),
and reads `cls.BAND_MARGIN` along its linearisation, `TickArrayBook`, `AggregateBook`, `object`:
64, bound in `AggregateBook` (F1).
The floor is $1000 - 64 = 936$ and the width $1010 - 1000 + 1 + 2 \cdot 64 = 139$ (F7).

**`again`.**
`wide.BAND_MARGIN = 0` binds an attribute of the instance (F1).
`wide.for_prices` is a class method reached through the instance,
and it receives the class of `wide`, `TickArrayBook`, not `wide` itself (F2).
`cls.BAND_MARGIN` is then searched in the classes alone,
by part i of Definition B.3.1 (`def.attributeReference`),
and the attribute of `wide` is never read: floor 936, width 139.

**`narrow`.**
`NarrowBook.for_prices` is the function of `TickArrayBook`, inherited,
and it receives `NarrowBook` (F2).
The linearisation of `NarrowBook` begins with `NarrowBook`, which binds `BAND_MARGIN = 0`,
found before the 64 of `AggregateBook` (F4): floor $1000$, width $1010 - 1000 + 1 = 11$.

In [ ]:
low, high = 1000, 1010
low - 64, high - low + 1 + 2 * 64, low - 0, high - low + 1 + 2 * 0

**The other choices.**
- (A) `1000 11 1000 11`: takes the class method to read the margin off the instance it is called on;
  it receives the class (F2).
- (C) `936 139 936 139`: takes `cls.BAND_MARGIN` to be searched from the class in which `for_prices` is written,
  `TickArrayBook`; it is searched from `cls`, here `NarrowBook`.
- (D) `1000 11 936 139`: the misreadings of (A) and (C) together:
  the margin of `again` read off the instance, and that of `narrow` searched from `TickArrayBook`.

## 5.4 · `question.bypassedWritesOnThreeBooks`

Bids 99 × 100 and 98 × 50 and an ask 101 × 100,
built by `from_levels` as an `AggregateBook`, a `CachedBestBook` and a `HeapBook`;
then 50 written at 100 and the level at 99 deleted, in the dictionary `bids` directly.

**Answer: (D).** `[100, 99, 98]`.

`from_levels` writes 99 and 98 through `set_size` (F7), and each index learns of them.
The two writes that follow go to the dictionary and not through `set_size`,
so no index is told of them.
The bid dictionary is now `{98: 50, 100: 50}`,
and hypothesis iii of Proposition B.6.2 (`prop.invariantByInduction`) fails on all three books:
the plain book keeps no index, and each subclass keeps one that no longer describes the dictionary.

- **`AggregateBook`** scans the dictionary (F5): $\max\{98, 100\} = 100$.
- **`CachedBestBook`** set its cache to 99 at the write of 99,
  and kept it at the write of 98, which is not better (F6).
  `best_price` returns the cache, 99, a price no longer in the book.
- **`HeapBook`** pushed $-99$ and $-98$ at the writes of 99 and 98, and never $-100$.
  `best_price` reads the top, 99, finds it missing from the dictionary and pops it: lazy deletion.
  The next entry, 98, is in the dictionary, and is returned.

**The other choices.**
- (A) `[100, 100, 100]`: takes the indexes to see the writes to the dictionary, as the scan does.
- (B) `[100, 99, 100]`: takes `HeapBook` to scan the dictionary;
  its candidates are the prices pushed by `set_size`, and 100 is not one of them.
- (C) `[100, 99, 99]`: misses the lazy deletion:
  the top of the heap is checked against the dictionary before it is returned.
- (E) `[100, 98, 98]`: credits the cache with the lazy deletion of the heap,
  discarding the stale 99 for the next price written through `set_size`;
  the cache holds one price, and `best_price` returns it unchecked.

## 5.5 · `question.lookupsCountedThroughTheInheritedMatching`

`CountingBook` overrides `best_price`, to count its calls.
Bid 99 × 100; asks 101 × 100, 102 × 80 and 105 × 20; market buys of 190 and of 40.

**Answer: (E).** `[2, 5, 7] 105`.

Every `self.best_price` in the code that `CountingBook` inherits,
the matching and the check of `from_levels` alike, is the override (F4).

- **`from_levels`.**
  Its writes go through `set_size`, which looks nothing up;
  `check_invariants` reads `best_bid_price` and `best_ask_price`, two calls of `self.best_price` (F7):
  2 lookups before any order.
- **The buy of 190** (F5).
  Pass 1 finds 101 and takes 100, leaving 90;
  pass 2 finds 102 and takes 80, leaving 10;
  pass 3 finds 105 and takes 10 of its 20.
  Nothing remains, and the loop ends without a fourth lookup: 3 lookups, 5 in all.
- **The buy of 40.**
  Pass 1 finds 105 and takes the 10 left there, leaving 30;
  pass 2 finds the ask side empty, and the loop breaks: 2 lookups, 7 in all.
  The 30 left rest at 105 on the bid side, the price the buy last traded at (F5).

The best bid is then $\max\{99, 105\} = 105$; the ask side is empty, and the book is not crossed.

In [ ]:
left_of_190 = 190 - 100 - 80
left_at_105 = 20 - left_of_190
left_of_190, left_at_105, 40 - left_at_105

**The other choices.**
- (A) `[0, 3, 5] 105`: misses the two lookups of `check_invariants` in `from_levels`.
- (B) `[0, 0, 0] 105`: takes `self.best_price` in the inherited code for the function of `AggregateBook`;
  by Proposition B.7.1 (`prop.lateBinding`) it is the override.
- (C) `[2, 5, 6] 105`: misses the lookup that finds the ask side empty and ends the second buy.
- (D) `[2, 5, 7] 99`: takes the 30 left of the second buy to be dropped;
  a market order rests its remainder at the price it last traded at (F5).

## 5.6 · `question.rescansCountedThroughTheCache`

`AuditedBook` overrides `_rescan` of `CachedBestBook`, to count its calls.
Bids 99 × 100 and 98 × 50; asks 101 × 100, 102 × 80 and 105 × 20; five messages.

**Answer: (B).** `2`.

`_best_after` is a function of `CachedBestBook`, and its call `self._rescan` is the override (F4).
It is made exactly when a write empties the cached best price (F6).
The writes of `from_levels` are all of positive sizes and empty nothing,
and they leave the cache at 99 on the bids and at 101 on the asks.

1. **Market buy of 150.**
   It takes the 100 at 101: the write of 0 at 101 empties the cached ask,
   and the rescan finds 102 (rescan 1).
   It then takes 50 at 102 and leaves 30, and the cache stands.
2. **Withdrawal of 20 at 105.**
   It empties 105, which is not the cached ask, 102.
3. **Withdrawal of 50 at 98.**
   It empties 98, which is not the cached bid, 99.
4. **Limit buy of 40 at 100.**
   The best ask, 102, lies above 100, so nothing trades (F5);
   the 40 rest at 100, better than the cached 99, and the cache becomes 100 without a scan.
5. **Market sell of 40.**
   It takes the 40 at 100, the cached bid:
   the write of 0 empties it, and the rescan finds 99 (rescan 2).

**The other choices.**
- (A) `0`: takes the call `self._rescan` inside `CachedBestBook` for its own function, not the override (F4).
- (C) `3`: takes every change of the cached price in the stream to cost a rescan,
  the limit buy that moves the bid to 100 among them;
  a better price replaces the cache directly.
- (D) `4`: takes every write that empties a level to cost a rescan, the two withdrawals among them;
  a level emptied behind the touch leaves the cache as it is.
- (E) `5`: takes every change of the cached price to cost a rescan,
  the two writes of `from_levels` that set the empty cache, to 99 and to 101, among them.

## 5.7 · `question.indexedBookWithoutSuper`

`IndexedBook` keeps the occupied prices of each side in a set, and finds its best prices there;
Alternative B is Alternative A without the call `super().set_size`.
Bids 99 × 100 and 98 × 50; asks 101 × 100 and 102 × 80; a market buy of 150.

**Answer: (C).** B prints `2 0`, then `None 0`.

**The book that B builds.**
B's `set_size` overrides that of `AggregateBook` and does not call it (F4),
so no write reaches the dictionaries:
they stay empty, while the sets hold $\{98, 99\}$ and $\{101, 102\}$.
The constructor does call `super().__init__`, so `bids` and `asks` exist,
Remark B.7.2 (`remark.constructorsBindTheState`),
and `check_invariants` passes: the best bid, 99, lies below the best ask, 101,
and no size is stored at all.

**The first line.**
`spread` reads the best prices off the sets: $101 - 99 = 2$.
`best_bid_size` is `size_at(BUY, 99)`, read from the empty dictionary: 0.

**The market buy** (F5).
Pass 1 finds 101, reads its size, 0, trades 0 and writes the size 0, which discards 101 from the set.
Pass 2 does the same at 102.
Pass 3 finds the ask side empty, and the loop breaks.
Nothing traded, so `last_traded_size` is 0, and the best ask is `None`.
`last_fill_price` is 102, so the 150 shares left rest at 102 on the bid side,
which in B adds 102 to the bid set and stores no size.
The second line is `None 0`.

**The other choices.**
- (A) `2 100`, then `102 150`: takes the function of the base class to run as well as the override;
  without `super().set_size` it does not run.
- (B) `2 0`, then `102 150`: sees the empty dictionary in `best_bid_size` and not in the matching,
  which reads its sizes with the same `size_at`.
- (D) `2 0`, and `apply` never returns: takes a trade of 0 to leave 101 in the set, so that every pass finds it again;
  the write of the size 0 discards it.
- (E) `AttributeError`: takes the dictionaries to be missing, as in `TickArrayBook`;
  the constructor of `IndexedBook` calls `super().__init__`, which binds them.

## 5.8 · `question.copyOfABookWithAVenue`

`VenueBook`, a subclass of `CachedBestBook` that adds a `venue`,
with the constructor `(venue, strict)` in A and `(strict, venue="MTA")` in B.
A book of the venue `"XETRA"` holding 100 bid at 99, copied.

**Answer: (A).** A raises `TypeError`; B prints `MTA 99`.

**The copy.**
`copy` and `_empty_like` are found in `AggregateBook` (F1),
and `_empty_like` returns `type(self)(self.strict)`:
it calls `VenueBook`, the class of the book, with one positional argument, `False` (F8).
- In A, `False` is bound to `venue`, and `strict` has no value: `TypeError` (F8),
  raised by `book.copy()`, before anything is printed.
- In B, `False` is bound to `strict`, and `venue` takes its default, `"MTA"`:
  the copy does not carry the venue of the book.

**The level.**
The copy writes the bid through `clone.set_size`,
which on a `VenueBook` is the function of `CachedBestBook` (F1, F4),
so the cache of the clone holds 99. B prints `MTA 99`.

**The other choices.**
- (B) Both `XETRA 99`: takes the copy to duplicate the book, attributes and all, without calling its class;
  `copy` calls the class through `_empty_like`, and writes only the levels.
- (C) A `TypeError`, B `XETRA 99`: reads the call of the class right,
  and takes `copy` to carry the venue onto the clone; the clone has the venue its constructor gave it.
- (D) A `False 99`: takes `strict` to fall back on a default, as in the constructor of `AggregateBook`;
  the constructor of A gives it none.
- (E) B `MTA None`: takes the copy to write past the cache;
  it writes through `clone.set_size`, the function of `CachedBestBook`.

## 5.9 · `question.buildingABookOfAGivenClass`

Three builders, each handed `CachedBestBook` and `TickArrayBook`, for a bid 99 × 100 and an ask 101 × 100.

**Answer: (D).**
`['CachedBestBook', 'TypeError']`, then `['AggregateBook', 'AggregateBook']`,
then `['CachedBestBook', 'TickArrayBook']`.

- **`build_a`** calls the class with `False`.
  `CachedBestBook(False)` is a book;
  `TickArrayBook(False)` binds `False` to `floor` and leaves `width`, which has no default, without a value:
  `TypeError` (F8).
- **`build_b`** calls `AggregateBook.for_prices`, a class method referenced on `AggregateBook`:
  it receives `AggregateBook` (F2) and returns `AggregateBook(False)` (F7), whatever `cls` is.
- **`build_c`** calls `cls.for_prices`.
  On `CachedBestBook` it finds the function of `AggregateBook`,
  which receives `CachedBestBook` and returns `CachedBestBook(False)`.
  On `TickArrayBook` it finds the override (F1),
  which computes the band from the prices 99 and 101: floor 35, width 131 (F7).
  Both levels lie in the band, at the slots 64 and 66.

This is the construction that Remark B.7.4 (`remark.limitsOfSubstitution`) prescribes
for code that builds a book of a class it is handed.

In [ ]:
floor, width = 99 - 64, 101 - 99 + 1 + 2 * 64
floor, width, 99 - floor, 101 - floor

**The other choices.**
- (A) Both classes on every line: misses that `TickArrayBook` needs a width,
  and that `build_b` names `AggregateBook`.
- (B) Both classes on the last two lines: takes `AggregateBook.for_prices` for `cls.for_prices`,
  so that `build_b` is `build_c`; referenced on `AggregateBook`, the class method receives `AggregateBook` (F2).
- (C) Both classes on the first line: misses that `TickArrayBook(False)` leaves the width without a value.
- (E) `TypeError` on the last line as well: takes `TickArrayBook.for_prices` for the function of `AggregateBook`,
  which calls `cls(strict)`; `TickArrayBook` overrides it.

## 5.10 · `question.fiveBooksOnPricesOutsideTheBand`

The five books, each built from a bid 99 × 100 and an ask 101 × 100,
each given, afresh, a limit sell of 10 at 166 and a limit buy of 10 at 34.

**Answer: (E).** `['TickArrayBook']`, then `['BitmapBook', 'TickArrayBook']`.

**The bands** (F7).
For the prices 99 and 101, `TickArrayBook` has the floor $99 - 64 = 35$,
the width $101 - 99 + 1 + 128 = 131$ and the ceiling $35 + 131 - 1 = 165$;
`BitmapBook` has the floor 35 and no ceiling.

**Neither order trades** (F5).
The sell at 166 lies above the best bid, 99, and the buy at 34 below the best ask, 101:
each loop breaks at its first pass, and the order rests at its own price:
`submit` reads the size there, 0, since a read outside the band returns zero, Section 1.2.4 (`sec.fasterBook`)
(`BitmapBook` reads its dictionaries),
and writes $0 + 10$ through `set_size`, which raises `ValueError` at a price the band cannot hold.
- The sell rests at 166.
  Its slot, $166 - 35 = 131$, is not below the width, and `TickArrayBook` raises;
  166 lies above the floor of `BitmapBook`, which accepts it.
- The buy rests at 34, below both floors, its slot $34 - 35 = -1$ negative: `BitmapBook` and `TickArrayBook` raise.

The three dictionary books accept any price,
and the names are listed in the order of `AXIS_B_VARIANTS`.
Each of the two banded books substitutes for `AggregateBook` only on the prices it can hold,
Remark B.7.4 (`remark.limitsOfSubstitution`).

In [ ]:
floor, width = 99 - 64, 101 - 99 + 1 + 2 * 64
floor, width, floor + width - 1, 166 - floor, 34 - floor

**The other choices.**
- (A) `[]` twice: takes every book to accept any price, as the dictionary books do.
- (B) `['TickArrayBook']` twice: takes `BitmapBook` to have no floor.
- (C) Both names on both lines: takes `BitmapBook` to have a ceiling as well.
- (D) `[]`, then both names: puts the ceiling at $p_0 + W = 166$, one tick too high.

## 5.11 · `question.cachedBestPriceWhenTheTouchEmpties`

A bid 1000 × 100 and $\ell$ asks of 100, from 1001 to $1000 + \ell$;
500 pairs of a market buy and a limit sell at 1001, of 10 shares in $S$ and of 100 in $E$;
`AggregateBook` against `CachedBestBook`, at $\ell = 10^3$ and $\ell = 10^4$.

**Answer: (D).**
On $S$ the time per message of `CachedBestBook` is about the same at both depths,
and that of `AggregateBook` grows about tenfold; on $E$ both grow about tenfold.

**One pair** (F5).
The market buy looks up the best ask, 1001, trades its whole size there, and stops.
The limit sell at 1001 looks up the best bid, 1000, which lies below it:
nothing trades, and the sell rests at 1001.
Each pair returns 1001 to 100 shares.

**`AggregateBook`.**
Every lookup of the best ask scans the $\ell$ ask prices, and the lookup of the best bid scans one (F5).
Each pair pays one scan of $\ell$ prices, on both streams,
and the scan grows tenfold from $\ell = 10^3$ to $\ell = 10^4$.

**`CachedBestBook`** (F6).
Its lookups read the cache.
- On $S$ the buy of 10 leaves 90 at 1001, and the cache stands;
  the sell writes 100 at 1001, no better than the cache, which stands.
  Nothing is scanned, and the time per message does not depend on $\ell$.
- On $E$ the buy of 100 empties 1001, the cached best ask, and `_rescan` scans the $\ell - 1$ asks left;
  the sell then writes 1001, better than the new cache, 1002, and restores it without a scan.
  Each pair pays one scan, as in `AggregateBook`.

The two books pass through the same configurations at different costs:
the cost of an operation lies outside eq. (B.7.1, `eq.commutingFold`),
Remark B.7.4 (`remark.limitsOfSubstitution`).
The verification notebook prints growths of 5 to 10 where a scan is paid,
and of about 1 for `CachedBestBook` on $S$;
the time a message spends outside the scan keeps the first below 10.

**The other choices.**
- (A) On $S$ neither changes, and on $E$ both grow: credits `AggregateBook` with the cache,
  taking its scan to be paid only when the touch empties; it scans at every lookup of the best ask (F5).
- (B) `CachedBestBook` constant on both streams: misses the rescan on $E$,
  where the cache is repaired by the scan that `AggregateBook` makes.
- (C) `AggregateBook` faster on both: takes every write to repair the cache by a scan,
  so that `CachedBestBook` pays two scans a pair where `AggregateBook` pays one;
  only a write that empties the cached price rescans (F6).
- (E) No change with $\ell$: takes a best price to cost the same at every depth;
  the scan of `AggregateBook` is $O(\ell)$, Table 1.4 (`tab.fasterBookCosts`).

## 5.12 · `question.abstractMethodRaisesAtInstantiation`

`TouchFrame`, a subclass of the abstract class `FrameSerializable` that binds `schema` and `to_frame` and not `from_frame`;
and `_HawkesState`, whose `step` raises `NotImplementedError`, on a baseline of total 1.0.

**Answer: (A).**
The first raises `TypeError` at the call of `TouchFrame`, and prints nothing;
the second prints `1.0`, then raises `NotImplementedError`.

**The first run.**
`FrameSerializable` declares three functions under `@abstractmethod`.
A class that leaves one of them without an override is abstract, and cannot be instantiated,
Section B.8 (`sec.abstractClasses`).
`TouchFrame` binds `schema` and `to_frame` and not `from_frame`,
so the call `TouchFrame(99, 101)` raises `TypeError` before any method runs, `to_json` among them.

**The second run.**
`_HawkesState` declares nothing under `@abstractmethod`, and the call creates it.
The state starts at $Z = 0$ (`unito26.lob.hawkes._HawkesState`), so `total_intensity` is the total baseline, $0.5 + 0.5 = 1.0$:
the first `print` prints `1.0`.
`events(10.0)` returns a generator, whose body runs at the call of `next`, a fact of Python:
`_crossing` is `None`, so it calls `self.step()`, the function of `_HawkesState`,
which raises `NotImplementedError`.
The second `print` is never called.

Section B.8 (`sec.abstractClasses`) and Remark B.8.1 (`remark.twoFormsOfAbstract`) give the two forms:
the abstract method is enforced at the call of the class, the raising body at the first event.

**The other choices.**
- (B) The table as JSON: takes an abstract method to matter only when it is called, as the raising `step` does;
  the check is made when the class is called.
- (C) `TypeError` at the call of `_HawkesState` as well:
  takes a body that raises `NotImplementedError` to make the class abstract; only `@abstractmethod` does.
- (D) The table as JSON, and `TypeError` at the call of `_HawkesState`: exchanges the two forms,
  taking the abstract declaration to act at the call of the method, which `to_json` never makes,
  and the raising `step` to make its class abstract.